In [23]:
import logging
import os

import ck_marketing.hunterio.hunter_api as cmhuhuap
import ck_marketing.linkedin.linkedin_utils as cmliprfi
import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint
from ck_marketing.hunterio.hunter_api import GoogleSheetsHelper, HunterIO
from ck_marketing.linkedin.phantombuster_api import Phantom

In [36]:
# Configure logger.
hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

# Print system signature.
_LOG.info("%s", henv.get_system_signature()[0])

# Configure the notebook style.
hprint.config_notebook()

DEBUG:helpers.hsystem:> (cd . && cd "$(git rev-parse --show-toplevel)/.." && (git rev-parse --is-inside-work-tree | grep -q true)) 2>&1
DEBUG:helpers.hsystem:> (git rev-parse --show-toplevel) 2>&1
DEBUG:helpers.hsystem:> (git branch --show-current) 2>&1
DEBUG:helpers.hsystem:> (git rev-parse --short HEAD) 2>&1
DEBUG:helpers.hsystem:> (git log --date=local --oneline --graph --date-order --decorate --pretty=format:'%h %<(8)%aN%  %<(65)%s (%>(14)%ar) %ad %<(10)%d' -3) 2>&1


INFO:__main__:# Git
  branch_name='CmTask10992__Build_pipeline_for_email_validation'
  hash='2d8544eb0'
  # Last commits:
    * 2d8544eb0 Shayan   AWS CloudWatch Synthetics Canaries documented (#10930)            (    9 days ago) Tue Dec 3 16:21:18 2024  (HEAD -> CmTask10992__Build_pipeline_for_email_validation, origin/master, origin/HEAD, master)
    * 6baf44a3c Nina Lee CmTask10927 Fix cmd in publish_master_trading_notebook            (    9 days ago) Tue Dec 3 14:41:15 2024           
    * f25cb3799 Nina Lee CMTask 10922 Fix build_run_notebook_cmd() call (#10924)           (    9 days ago) Tue Dec 3 13:58:07 2024           
# Machine info
  system=Linux
  node name=5b818430e535
  release=5.15.0-1072-aws
  version=#78~20.04.1-Ubuntu SMP Wed Oct 9 15:30:47 UTC 2024
  machine=x86_64
  processor=x86_64
  cpu count=8
  cpu freq=scpufreq(current=2499.998, min=0.0, max=0.0)
  memory=svmem(total=33280225280, available=24316379136, percent=26.9, used=8473686016, free=4250132480, active=3888

# Clean Profiles

In [37]:
hunter_api_key = os.getenv("Hunter_API_KEY")

In [30]:
# Google Drive Setup.
google_creds_path = "service.json"
google_sheet_helper = GoogleSheetsHelper(google_creds_path)
file_id = "1UOov0ZmZoCVBIUg825k9LgT8m6p43aW91aR2WmJNDcg"

In [31]:
df = google_sheet_helper.read_sheet(file_id)

INFO:root:Opened spreadsheet with ID: 1UOov0ZmZoCVBIUg825k9LgT8m6p43aW91aR2WmJNDcg
INFO:root:Selected worksheet: Sheet1


In [32]:
df.head()

,Person - Name,Person - Email - Work,Person - Email - Home,Person - Email - Other,Person - Phone - Work,Person - Phone - Home,Person - Phone - Mobile,Person - Phone - Other
0,A Gadala,a.gadala@gbh.com.do,,,,,,
1,A Guzman,a.guzman@gbh.com.do,,,,,,
2,A.M. Saad,ams@btbmed.com,,,,,,
3,Aadeel Akhtar,aakhtar@psyonic.co,,,,,,
4,Aage Lauridsen,aage.lauridsen@enterahealth.com,,,(515) 289-7652,,,


# Verify Email

In [33]:
hunter_instance = HunterIO(hunter_api_key)
verified_df = hunter_instance.verify_emails(df, "Person - Email - Work")

In [35]:
sheet = google_sheet_helper.google_account.open_by_key(file_id)
cleaned_profiles_tab = sheet.add_worksheet(
    title="hunter_verification", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, verified_df, "hunter_verification")

INFO:ck_marketing.hunterio.hunter_apii:Email extraction completed. Results saved in the new tab: hunter_verification


In [34]:
verified_df.head()

,Person - Name,Person - Email - Work,Person - Email - Home,Person - Email - Other,Person - Phone - Work,Person - Phone - Home,Person - Phone - Mobile,Person - Phone - Other,hunter_verification
0,A Gadala,a.gadala@gbh.com.do,,,,,,,valid
1,A Guzman,a.guzman@gbh.com.do,,,,,,,valid
2,A.M. Saad,ams@btbmed.com,,,,,,,invalid
3,Aadeel Akhtar,aakhtar@psyonic.co,,,,,,,valid
4,Aage Lauridsen,aage.lauridsen@enterahealth.com,,,(515) 289-7652,,,,valid
